# The RL loop: train on rewards

In this tutorial you train a model with reinforcement learning on a few math problems. Each problem is a task: what the model is asked, and a verifier that scores its answer. The cookbook's loop, `train_rl.train`, attempts every task of a batch several times on the run's own sampler, trains on the rewards and publishes each step back to the sampler. Problems the run never trains measure it before and after.

[The SFT loop](301_sft_loop.ipynb) introduces the cookbook's loops, and [Your first RL run](104_first_rl.ipynb) writes one RL step out by hand, with the SDK alone. This notebook uses the cookbook's package, `looper_cookbook`. Until that is published, run it from a checkout of the repository, in its environment. Put your API key in the settings cell's `API_KEY`.

## Settings

The model, its tokenizer, where Looper is, your project and your API key. Then the run: a LoRA's rank and its learning rate, a turn's token budget, how many problems a step takes, how many attempts at each it compares, how many steps, and how many problems are held out.

In [ ]:
MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = "Qwen/Qwen3.5-9B"
BASE_URL = "https://api.looper.goaly.ai"
PROJECT = "example"
API_KEY = "YOUR_API_KEY"
LORA_RANK = 32
LEARNING_RATE = 8e-5
MAX_TURN_TOKENS = 512
BATCH_SIZE = 8
GROUP_SIZE = 8
STEPS = 8
HELD_OUT = 8

## Problems to tasks

The example file holds 32 grade-school math problems from GSM8K's train split, each with its answer, a number. A task is one problem as the loop attempts it: the messages the model is asked, and the config of a verifier that scores an attempt. Here the question asks for the answer after `####`, where GSM8K's own worked answers put it, and `FinalNumber` gives 1 when the number there is right, else 0.

A task is the config of an episode session, which is one attempt at it. `AgentSession` is the cookbook's session. With no environment, an attempt is one turn: the model's answer.

`TaskDataset` puts the training tasks in batches, one per step. The last `HELD_OUT` problems are kept out of it.

In [ ]:
import json
from importlib import resources

from transformers import AutoTokenizer

from looper_cookbook import train_rl
from looper_cookbook.datasets.tasks import TaskDataset
from looper_cookbook.episode_sessions.agent_session import AgentSession
from looper_cookbook.message_formatters.models import message_formatter
from looper_cookbook.verifiers.final_number import FinalNumber
from looper_sdk import AdamW, Lora, SamplingParams, ServiceClient, TrainingConfig

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
formatter = message_formatter(MODEL, tokenizer, enable_thinking=False)
path = resources.files("looper_cookbook") / "example_data" / "math_problems.jsonl"
problems = [json.loads(line) for line in path.read_text().splitlines()]
INSTRUCTION = "Solve it step by step, then give the final answer as a number after '####'."


def asked(problem):
    return [{"role": "user", "content": f"{problem['question']}\n{INSTRUCTION}"}]


# An attempt here is one turn, so an episode's token budget is the longest prompt
# and a turn's budget. The run and its sampler are sized to hold it.
prompts = [formatter.build_generation_prompt(asked(problem)) for problem in problems]
episode_tokens = max(len(prompt) for prompt in prompts) + MAX_TURN_TOKENS


def task(problem):
    return AgentSession.Config(
        messages=asked(problem),
        verifier=FinalNumber.Config(answer=str(problem["answer"])),
        max_episode_tokens=episode_tokens,
    )


tasks = [task(problem) for problem in problems]
train_tasks, held_out = tasks[:-HELD_OUT], tasks[-HELD_OUT:]
dataset = TaskDataset(train_tasks, batch_size=BATCH_SIZE, seed=0)
print(f"{len(train_tasks)} tasks in {len(dataset)} batches a pass, and {len(held_out)} held out")
print(f"an episode, a prompt and its one turn, holds at most {episode_tokens} tokens")

## The run and its sampler

`TrainingConfig` names the model, the LoRA and the optimizer: every step trains at `LEARNING_RATE`. The run's sizes pick its GPUs: `max_seq_len` holds an episode's token budget, and `tokens_per_step` is what one step trains.

A run trains. To sample its own weights it needs a sampler of its own, which its training client creates: `training_client.create_sampling_client(...)`. Its sizes pick the sampler's GPUs: the longest sequence, and how many turns it runs at once. It returns once the sampler is ready and linked to the run's trainer, and the sampler serves nothing until the run publishes.

In [ ]:
config = ServiceClient.Config(
    base_url=BASE_URL, project=PROJECT, api_key=API_KEY, display_name="rl-loop"
)
training = TrainingConfig(
    model=MODEL,
    kind=Lora(rank=LORA_RANK),
    tokens_per_step=BATCH_SIZE * GROUP_SIZE * episode_tokens,
    max_seq_len=episode_tokens,
    optimizer=AdamW(learning_rate=LEARNING_RATE, betas=(0.9, 0.95), max_grad_norm=1.0),
)


async def start(service):
    print("starting the run and its sampler...")
    training_client = await service.create_training_client(training, timeout_s=20 * 60)
    sampler = await training_client.create_sampling_client(
        max_seq_len=episode_tokens,
        max_concurrent_sequences=BATCH_SIZE * GROUP_SIZE,
        timeout_s=20 * 60,
    )
    return training_client, sampler

## A held-out measurement

The loop measures the model as well as training it. Hand it the held-out tasks as `eval_tasks`, and it attempts each of them once on the run's sampler: before its first step, which is the baseline, and after its last. Nothing it samples for them trains.

It reports the mean reward, here the share of the held-out problems answered right, as `eval/reward`, with its standard error as `eval/reward_stderr`. It samples those attempts greedily, at temperature 0, unless the config's `eval_params` says otherwise, so it measures the model's best answer.

## The loop

`train_rl.train` takes the run, its sampler, the dataset and the loop's config. Each step it attempts every task of the step's batch `GROUP_SIZE` times, all at once, and sends each task's attempts to the trainer as one group: `Grpo` compares each attempt's reward with the rest of its group. Then it takes the optimizer step and publishes it. It publishes before its first sample and awaits each publish before it samples again, so every step trains on answers of the step before it.

It calls `on_step` with each report: the run's result, the version the step's answers came from, and a flat map of metrics. With `eval_tasks`, its first report is of the step it started at, before it trained anything.

A group whose attempts all score the same trains nothing: `Grpo` learns from differences within a group. The report's `groups_trained` counts the groups that differed.

The session holds the run and its sampler. Closing it, as `async with` does on the way out, ends both.

In [ ]:
loop_config = train_rl.Config(
    params=SamplingParams(max_tokens=MAX_TURN_TOKENS, temperature=1.0), group_size=GROUP_SIZE
)


def report(step):
    if step.result is None:
        print(f"before training: held out {step.metrics['eval/reward']:.3f}")
        return
    line = (
        f"step {step.step}/{STEPS}: reward {step.metrics['reward']:.3f} "
        f"on answers of step {step.version.step}, "
        f"{step.metrics['groups_trained']:.0f} of {step.metrics['groups']:.0f} groups trained"
    )
    if "eval/reward" in step.metrics:
        line += f", held out {step.metrics['eval/reward']:.3f}"
    print(line)


async def train_on_rewards():
    async with await config.build() as service:
        training_client, sampler = await start(service)
        return await train_rl.train(
            training_client,
            sampler,
            dataset,
            loop_config,
            formatter=formatter,
            steps=STEPS,
            eval_tasks=held_out,
            on_step=report,
        )


reports = None
reports = await train_on_rewards()

## What the loop does for you

`looper_cookbook/train_rl.py` is short enough to read, and meant to be copied and changed when your training needs something else. It gets five things right that a loop of your own also has to:

- **It samples the weights it just trained.** A sample submitted before a publish lands comes from the weights before it. The loop awaits each publish before it samples again, and each report's `version` shows which weights its answers came from.
- **A group goes whole.** A task's attempts are one group, in one call, so the trainer forms each attempt's advantage from its own group's rewards. A datum never carries an advantage.
- **Every attempt is in flight at once.** The sampler batches a step's turns on its GPUs. Its `max_concurrent_sequences` bounds how many run together, and the rest wait in its queue.
- **The run's step decides the data.** Step `k` trains `dataset.batch(k)`, so the loop called again on the same run goes on from where the run stands.
- **It checks each result's step,** and raises if another writer or a rollback moved the run.

## Next steps

The same loop over all of GSM8K's train split, measured on its test split, runs as a script, with flags for the model, the sizes and the steps:

```bash
python -m looper_cookbook.recipes.rl_basic --steps 20
```